In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("../data/MachineLearningCSV/MachineLearningCVE")

print("Dataset folder exists:", DATA.exists())

frames = []

for f in sorted(DATA.glob("*.csv")):
    d = pd.read_csv(f, encoding="latin1")
    d.columns = d.columns.str.strip()
    d["source_file"] = f.name
    frames.append(d)

print("Files loaded:", len(frames))

raw = pd.concat(frames, ignore_index=True)

print("Dataset shape:", raw.shape)
raw["Label"].value_counts()

Dataset folder exists: True
Files loaded: 8
Dataset shape: (2830743, 80)


Label
BENIGN                          2273097
DoS Hulk                         231073
PortScan                         158930
DDoS                             128027
DoS GoldenEye                     10293
FTP-Patator                        7938
SSH-Patator                        5897
DoS slowloris                      5796
DoS Slowhttptest                   5499
Bot                                1966
Web Attack ï¿½ Brute Force         1507
Web Attack ï¿½ XSS                  652
Infiltration                         36
Web Attack ï¿½ Sql Injection         21
Heartbleed                           11
Name: count, dtype: int64

In [2]:
def family(label):
    l = str(label).strip()

    if l == "BENIGN":
        return "benign"
    if l == "PortScan":
        return "portscan"
    if l in ("FTP-Patator", "SSH-Patator"):
        return "bruteforce"
    if l.startswith("DoS") or l == "DDoS":
        return "dos"
    if l.startswith("Web Attack"):
        return "web"

    return "other"

raw["family"] = raw["Label"].map(family)

raw["family"].value_counts()

family
benign        2273097
dos            380688
portscan       158930
bruteforce      13835
web              2180
other            2013
Name: count, dtype: int64

In [3]:
CIC_RENAME = {
    "Destination Port": "dst_port",
    "Flow Duration": "duration_us",
    "Total Fwd Packets": "fwd_pkts",
    "Total Backward Packets": "bwd_pkts",
    "Total Length of Fwd Packets": "fwd_bytes",
    "Total Length of Bwd Packets": "bwd_bytes",
    "Flow Bytes/s": "bytes_per_s",
    "Flow Packets/s": "pkts_per_s",
    "SYN Flag Count": "syn_count",
    "RST Flag Count": "rst_count",
    "FIN Flag Count": "fin_count",
}

def cic_to_common(df):
    df = df.rename(columns=lambda c: c.strip())

    out = df[list(CIC_RENAME.keys())].rename(
        columns=CIC_RENAME
    )

    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out.replace([np.inf, -np.inf], np.nan)

In [4]:
CIC_RENAME = {
    "Destination Port": "dst_port",
    "Flow Duration": "duration_us",
    "Total Fwd Packets": "fwd_pkts",
    "Total Backward Packets": "bwd_pkts",
    "Total Length of Fwd Packets": "fwd_bytes",
    "Total Length of Bwd Packets": "bwd_bytes",
    "Flow Bytes/s": "bytes_per_s",
    "Flow Packets/s": "pkts_per_s",
    "SYN Flag Count": "syn_count",
    "RST Flag Count": "rst_count",
    "FIN Flag Count": "fin_count",
}

def cic_to_common(df):
    df = df.rename(columns=lambda c: c.strip())

    out = df[list(CIC_RENAME.keys())].rename(
        columns=CIC_RENAME
    )

    for col in out.columns:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out.replace([np.inf, -np.inf], np.nan)

In [5]:
feats = cic_to_common(raw)

feats["family"] = raw["family"].to_numpy()
feats["source_file"] = raw["source_file"].to_numpy()

before_other = len(feats)
feats = feats[feats["family"] != "other"].copy()
after_other = len(feats)

before_missing = len(feats)
feats = feats.dropna().copy()
after_missing = len(feats)

before_duplicates = len(feats)
feats = feats.drop_duplicates(
    subset=feats.columns.difference(["source_file"])
).copy()
after_duplicates = len(feats)

print("Rows before cleaning:", before_other)
print("Other-class rows removed:", before_other - after_other)
print("Rows with missing/infinite values removed:", before_missing - after_missing)
print("Duplicate rows removed:", before_duplicates - after_duplicates)
print("Final rows:", len(feats))

feats["family"].value_counts()

Rows before cleaning: 2830743
Other-class rows removed: 2013
Rows with missing/infinite values removed: 2857
Duplicate rows removed: 551187
Final rows: 2274686


family
benign        1860749
dos            313824
portscan        90618
bruteforce       7406
web              2089
Name: count, dtype: int64

In [6]:
cols = [
    c for c in feats.columns
    if c not in ("family", "source_file")
]

conflicts = feats.groupby(cols)["family"].nunique()

conflict_count = int((conflicts > 1).sum())

print("Conflicting feature rows:", conflict_count)

Conflicting feature rows: 1429


In [8]:
OUTPUT = Path("../data/processed")
OUTPUT.mkdir(parents=True, exist_ok=True)

feats["is_attack"] = (
    feats["family"] != "benign"
).astype("int8")

output_file = OUTPUT / "cic_common.pkl"
feats.to_pickle(output_file)

print("Saved to:", output_file.resolve())
print("Final shape:", feats.shape)
print("\nClass counts:")
print(feats["family"].value_counts())

print("\nBinary label counts:")
print(feats["is_attack"].value_counts())

Saved to: C:\Users\Harman\Lumen-IDS\data\processed\cic_common.pkl
Final shape: (2274686, 14)

Class counts:
family
benign        1860749
dos            313824
portscan        90618
bruteforce       7406
web              2089
Name: count, dtype: int64

Binary label counts:
is_attack
0    1860749
1     413937
Name: count, dtype: int64
